
# Bengali NLP Safety — V17 Lite

This is the **lightweight replacement** for the previous V17 notebook.

The old notebook embedded ~MBs of data directly inside a code cell, which can make the
Colab browser tab freeze while rendering/syntax-highlighting the notebook.

### Run this version

1. Set **Runtime → Change runtime type → T4 GPU**.
2. Run cell 1.
3. When prompted, upload **`v17_data.zip`**.
4. Continue with **Run all**.
5. At the end, download **`v17_results.zip`** and send it to ChatGPT.

Current protected Kaggle champion: **0.586**.


In [ ]:

# 1. Upload the lightweight data bundle
from google.colab import files
import zipfile, pathlib, os

uploaded = files.upload()
assert "v17_data.zip" in uploaded, "Please upload the supplied v17_data.zip file."

DATA_DIR = pathlib.Path("/content/v17_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v17_data.zip","r") as z:
    z.extractall(DATA_DIR)

print("Extracted files:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes")


In [ ]:

# 1. Settings
SEED = 42
N_FOLDS = 5
TARGET = "is_toxic"
ALL_LABELS = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]

EPOCHS = 4
SNAPSHOT_EPOCHS = [3,4]
TRAIN_BATCH_SIZE = 8
ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

MODELS = [
    {
        "name":"banglabert_toxic",
        "model":"csebuetnlp/banglabert",
        "view":"v2_text",
        "max_length":128,
        "lr":1.5e-5,
    },
    {
        "name":"xlmr_toxic",
        "model":"FacebookAI/xlm-roberta-base",
        "view":"clean_text",
        "max_length":128,
        "lr":1.5e-5,
    },
]

# Conservative replacement gate.
MIN_CROSSFIT_GAIN = 0.003
MIN_WINNING_FOLDS = 3
MAX_LOSING_FOLDS = 1

print("Models:", [x["name"] for x in MODELS])


In [ ]:

# 2. Install dependencies
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])
print("Installation complete.")


In [ ]:

# 4. Imports and data
import gc, re, math, random, json, zipfile, unicodedata, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU in Colab."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2_valid = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2_test = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
valid_folds = pd.read_csv(DATA_DIR/"validation_folds.csv", dtype={"ID":str})
champion = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})

with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z:
    old_xlmr_valid = z["valid"].copy()
    old_xlmr_test = z["test"].copy()

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert champion["ID"].tolist()==test["ID"].tolist()
assert v2_valid["ID"].tolist()==valid["ID"].tolist()
assert v2_test["ID"].tolist()==test["ID"].tolist()

def clean_text(x):
    return re.sub(r"\s+"," ",unicodedata.normalize("NFC",str(x))).strip()

def v2_text(x):
    return re.sub(r"\s+"," ",str(x).replace("\u200c"," ").replace("\u200d"," ")).strip()

combined = pd.concat([train,valid],ignore_index=True)
for frame in [combined,test]:
    frame["clean_text"] = frame["text"].map(clean_text)
    frame["v2_text"] = frame["text"].map(v2_text)

print("train/valid/combined/test:", len(train), len(valid), len(combined), len(test))
print("Combined is_toxic positives:", int(combined[TARGET].sum()), "/", len(combined))


In [ ]:

# 5. Guaranteed balanced grouped 5-fold assignment
def normalize_group_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    return re.sub(r"\s+"," ",x).strip().lower()

combined["group_text"] = combined["text"].map(normalize_group_text)

# Balance all seven labels so correlated toxicity subtypes are spread across folds.
group_stats = combined.groupby("group_text",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in ALL_LABELS}
).reset_index()

global_pos = np.array([combined[l].sum() for l in ALL_LABELS],dtype=float)
total_rows = float(len(combined))
rarity = 1.0/np.sqrt(np.maximum(global_pos,1.0))
mat = group_stats[[f"sum_{l}" for l in ALL_LABELS]].to_numpy(float)

priority = (mat*rarity).sum(axis=1) + .005*group_stats["size"].to_numpy()
rng = np.random.default_rng(SEED)
order = np.argsort(-(priority + rng.random(len(group_stats))*1e-6))

fold_pos = np.zeros((N_FOLDS,len(ALL_LABELS)),dtype=float)
fold_size = np.zeros(N_FOLDS,dtype=float)
assignment={}

for idx in order:
    row=group_stats.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in ALL_LABELS],dtype=float)
    size=float(row["size"])
    costs=[]
    for f in range(N_FOLDS):
        pos_after=fold_pos.copy()
        size_after=fold_size.copy()
        pos_after[f]+=vec
        size_after[f]+=size
        label_share=pos_after/np.maximum(global_pos[None,:],1.0)
        size_share=size_after/total_rows
        label_cost=np.mean((label_share-1/N_FOLDS)**2)
        size_cost=np.mean((size_share-1/N_FOLDS)**2)
        costs.append(label_cost+.35*size_cost)
    best=int(np.argmin(costs))
    assignment[row["group_text"]]=best
    fold_pos[best]+=vec
    fold_size[best]+=size

combined["fold"]=combined["group_text"].map(assignment).astype(int)

fold_counts=combined.groupby("fold").size().reindex(range(N_FOLDS),fill_value=0)
positive_table=combined.groupby("fold")[ALL_LABELS].sum().reindex(range(N_FOLDS),fill_value=0)

assert combined.groupby("group_text")["fold"].nunique().max()==1
assert (fold_counts>0).all()
assert fold_counts.min() >= .8*(len(combined)/N_FOLDS)
assert fold_counts.max() <= 1.2*(len(combined)/N_FOLDS)
assert (positive_table[TARGET]>0).all()

print("✅ Fold checks passed")
print("Fold sizes:",fold_counts.to_dict())
print("is_toxic positives:",positive_table[TARGET].to_dict())


In [ ]:

# 6. Dataset / inference helpers
class ToxicDataset(Dataset):
    def __init__(self, frame, tokenizer, view, max_length, with_labels=True):
        self.texts=frame[view].tolist()
        self.labels=frame[TARGET].to_numpy(np.float32) if with_labels else None
        self.tokenizer=tokenizer
        self.max_length=max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self,i):
        enc=self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        if self.labels is not None:
            enc["labels"]=self.labels[i]
        return enc

class Collator:
    def __init__(self,tokenizer):
        self.tokenizer=tokenizer
    def __call__(self,items):
        labels=None
        if "labels" in items[0]:
            labels=torch.tensor([x.pop("labels") for x in items],dtype=torch.float32)
        batch=self.tokenizer.pad(items,padding=True,pad_to_multiple_of=8,return_tensors="pt")
        if labels is not None:
            batch["labels"]=labels
        return batch

@torch.no_grad()
def predict(model,loader):
    assert len(loader.dataset)>0
    model.eval()
    parts=[]
    for batch in loader:
        batch.pop("labels",None)
        inputs={k:v.to("cuda") for k,v in batch.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inputs).logits.squeeze(-1)
        parts.append(torch.sigmoid(logits.float()).cpu().numpy())
    assert parts
    return np.concatenate(parts)


In [ ]:

# 7. Train 5 OOF folds for each model family
RUN_DIR=Path("/content/v17_output")
RUN_DIR.mkdir(exist_ok=True)

fold_seeds=[42,3407,2026,17,1337]
model_results={}

for cfg in MODELS:
    name=cfg["name"]
    print("\n"+"#"*78)
    print("MODEL:",name)
    oof=np.zeros(len(combined),dtype=np.float32)
    test_fold_probs=[]
    history=[]

    for fold in range(N_FOLDS):
        tr=combined[combined["fold"]!=fold].copy()
        va=combined[combined["fold"]==fold].copy()
        assert len(tr)>0 and len(va)>0
        assert set(tr["group_text"]).isdisjoint(set(va["group_text"]))

        cache=RUN_DIR/f"{name}_fold_{fold}.npz"
        history_path=RUN_DIR/f"{name}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache,allow_pickle=False) as z:
                idx=z["va_index"]
                vp=z["valid"]
                tp=z["test"]
            if np.array_equal(idx,va.index.to_numpy()) and len(vp)==len(va) and len(tp)==len(test):
                print(f"✅ Reusing {name} fold {fold}")
                oof[idx]=vp
                test_fold_probs.append(tp)
                if history_path.exists():
                    history.extend(pd.read_csv(history_path).to_dict("records"))
                continue

        print(f"\n{name} fold {fold}: train={len(tr)} valid={len(va)}")
        seed_everything(fold_seeds[fold])

        tokenizer=AutoTokenizer.from_pretrained(cfg["model"])
        collator=Collator(tokenizer)
        tr_loader=DataLoader(
            ToxicDataset(tr,tokenizer,cfg["view"],cfg["max_length"],True),
            batch_size=TRAIN_BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True,collate_fn=collator
        )
        va_loader=DataLoader(
            ToxicDataset(va,tokenizer,cfg["view"],cfg["max_length"],True),
            batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=collator
        )
        te_loader=DataLoader(
            ToxicDataset(test,tokenizer,cfg["view"],cfg["max_length"],False),
            batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=collator
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            cfg["model"],num_labels=1
        ).to("cuda")

        # Macro-F1 gives negative and positive classes equal importance.
        # Mild square-root balancing avoids an aggressive class-weight shift.
        pos=float(tr[TARGET].sum())
        neg=float(len(tr)-pos)
        pos_weight=np.sqrt(neg/max(pos,1.0))
        loss_fn=torch.nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight],dtype=torch.float32,device="cuda")
        )
        print("pos_weight:",round(pos_weight,4))

        no_decay=["bias","LayerNorm.weight","layer_norm.weight"]
        groups=[
            {"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],
             "weight_decay":WEIGHT_DECAY},
            {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],
             "weight_decay":0.0},
        ]
        optimizer=torch.optim.AdamW(groups,lr=cfg["lr"])
        steps=math.ceil(len(tr_loader)/ACCUM_STEPS)*EPOCHS
        scheduler=get_linear_schedule_with_warmup(
            optimizer,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")
        val_snaps=[]
        test_snaps=[]
        fold_hist=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            running=0.0
            seen=0

            for step,batch in enumerate(tqdm(tr_loader,desc=f"{name} fold {fold} epoch {epoch}/{EPOCHS}")):
                y=batch.pop("labels").to("cuda")
                inputs={k:v.to("cuda") for k,v in batch.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inputs).logits.squeeze(-1)

                loss=loss_fn(logits.float(),y)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite loss")

                scaler.scale(loss/ACCUM_STEPS).backward()
                running+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM_STEPS==0 or step+1==len(tr_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    old_scale=scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale()>=old_scale:
                        scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            rec={"model":name,"fold":fold,"epoch":epoch,"loss":running/max(seen,1)}
            if epoch in SNAPSHOT_EPOCHS:
                val_snaps.append(predict(model,va_loader))
                test_snaps.append(predict(model,te_loader))
            fold_hist.append(rec)
            pd.DataFrame(fold_hist).to_csv(history_path,index=False)
            print(rec)

        vp=np.mean(val_snaps,axis=0)
        tp=np.mean(test_snaps,axis=0)
        assert len(vp)==len(va) and len(tp)==len(test)
        assert np.isfinite(vp).all() and np.isfinite(tp).all()

        oof[va.index.to_numpy()]=vp
        test_fold_probs.append(tp)
        history.extend(fold_hist)

        np.savez_compressed(
            cache,va_index=va.index.to_numpy(),valid=vp,test=tp
        )

        del model,tokenizer,tr_loader,va_loader,te_loader,optimizer,scheduler,scaler
        gc.collect()
        torch.cuda.empty_cache()

    assert len(test_fold_probs)==N_FOLDS
    model_test=np.mean(test_fold_probs,axis=0)
    model_results[name]={"oof":oof,"test":model_test}
    np.savez_compressed(RUN_DIR/f"{name}_oof_test.npz",oof=oof,test=model_test)
    pd.DataFrame(history).to_csv(RUN_DIR/f"{name}_training_history.csv",index=False)

print("\n✅ Both 5-fold model families completed.")


In [ ]:

# 8. Reconstruct the proven V14/V8 is_toxic baseline probabilities
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))
def sigmoid(z):
    return 1/(1+np.exp(-z))

v2v=v2_valid["is_toxic_prob"].to_numpy(float)
v2t=v2_test["is_toxic_prob"].to_numpy(float)

# V8/V14 toxic model:
# 87.5% V2 + 12.5% old XLM-R mean epochs 5/6, probability-space blend.
old_xl_v=old_xlmr_valid[1:3,:,0].mean(axis=0)
old_xl_t=old_xlmr_test[1:3,:,0].mean(axis=0)

baseline_valid=.875*v2v + .125*old_xl_v
baseline_test=.875*v2t + .125*old_xl_t
baseline_threshold=0.6850110311061144

# Original 500 validation rows correspond to the tail of combined.
orig_valid_idx=np.arange(len(train),len(train)+len(valid))
bang_valid=model_results["banglabert_toxic"]["oof"][orig_valid_idx]
xlmr_valid=model_results["xlmr_toxic"]["oof"][orig_valid_idx]
bang_test=model_results["banglabert_toxic"]["test"]
xlmr_test=model_results["xlmr_toxic"]["test"]

fold_map=valid_folds.set_index("ID")["fold"]
selection_folds=valid["ID"].map(fold_map).to_numpy(int)
assert set(selection_folds)==set(range(N_FOLDS))

y=valid[TARGET].to_numpy(int)
print("Baseline toxic positives on validation:",int((baseline_valid>=baseline_threshold).sum()))


In [ ]:

# 9. Correct binary-macro-F1 cross-fitted threshold search
def binary_macro(y_true,y_pred):
    return float(f1_score(y_true,y_pred,average="macro",labels=[0,1],zero_division=0))

def best_threshold_macro(y_true,p):
    p=np.asarray(p,float)
    # Candidate thresholds are probability values plus edge guards.
    vals=np.unique(p)
    if len(vals)>600:
        vals=np.quantile(vals,np.linspace(0,1,600))
    candidates=np.unique(np.r_[0.0, vals, 1.000001])
    scores=[binary_macro(y_true,p>=t) for t in candidates]
    best=max(scores)
    # Prefer the median threshold among exact/near ties for stability.
    tied=candidates[np.isclose(scores,best,atol=1e-12,rtol=0)]
    return float(np.median(tied))

def crossfit(p):
    pred=np.zeros(len(y),dtype=np.int8)
    fs=[]
    ts=[]
    for fold in range(N_FOLDS):
        held=selection_folds==fold
        t=best_threshold_macro(y[~held],p[~held])
        ts.append(t)
        pred[held]=(p[held]>=t).astype(np.int8)
        fs.append(binary_macro(y[held],pred[held]))
    return {
        "score":binary_macro(y,pred),
        "fold_scores":np.asarray(fs,float),
        "thresholds":np.asarray(ts,float),
        "prediction":pred,
    }

baseline_cv=crossfit(baseline_valid)
print("V14 toxic baseline crossfit macro-F1:",baseline_cv["score"])
print("fold scores:",np.round(baseline_cv["fold_scores"],4))
print("fold thresholds:",np.round(baseline_cv["thresholds"],4))


In [ ]:

# 10. Search conservative 3-way ensembles:
# baseline + new BanglaBERT + new XLM-R
rows=[]
best=None

for mode in ["prob","logit"]:
    # Keep at least 40% of the proven baseline.
    for wb_i in range(8,21):  # 0.40 ... 1.00
        wb=wb_i/20
        remaining=1-wb
        # Split remaining mass between the two new OOF models.
        steps=int(round(remaining*20))
        for k in range(steps+1):
            w_bang=k/20
            w_xl=remaining-w_bang
            if w_xl < -1e-12:
                continue

            if mode=="prob":
                p=wb*baseline_valid+w_bang*bang_valid+w_xl*xlmr_valid
            else:
                p=sigmoid(
                    wb*logit(baseline_valid)+
                    w_bang*logit(bang_valid)+
                    w_xl*logit(xlmr_valid)
                )

            cv=crossfit(p)
            wins=int((cv["fold_scores"]>baseline_cv["fold_scores"]+1e-12).sum())
            losses=int((cv["fold_scores"]<baseline_cv["fold_scores"]-1e-12).sum())
            ties=N_FOLDS-wins-losses
            gain=cv["score"]-baseline_cv["score"]
            passes=(
                gain>=MIN_CROSSFIT_GAIN and
                wins>=MIN_WINNING_FOLDS and
                losses<=MAX_LOSING_FOLDS
            )

            rec={
                "mode":mode,
                "w_baseline":wb,
                "w_banglabert":w_bang,
                "w_xlmr":w_xl,
                "crossfit_macro_f1":cv["score"],
                "baseline_macro_f1":baseline_cv["score"],
                "gain":gain,
                "wins":wins,
                "ties":ties,
                "losses":losses,
                "passes_gate":passes,
                "median_threshold":float(np.median(cv["thresholds"])),
            }
            rows.append(rec)

            key=(passes,cv["score"],wins,-losses,wb)
            if best is None or key>best["key"]:
                best={"key":key,"rec":rec,"cv":cv}

search=pd.DataFrame(rows).sort_values(
    ["passes_gate","gain","wins","losses","w_baseline"],
    ascending=[False,False,False,True,False]
)
search.to_csv(RUN_DIR/"toxic_ensemble_search.csv",index=False)
display(search.head(25).round(5))

print("\nBEST:")
print(best["rec"])


In [ ]:

# 11. Build a controlled V17 submission only if the gate passes
rec=best["rec"]
passes=bool(rec["passes_gate"])

if rec["mode"]=="prob":
    best_test=(
        rec["w_baseline"]*baseline_test+
        rec["w_banglabert"]*bang_test+
        rec["w_xlmr"]*xlmr_test
    )
else:
    best_test=sigmoid(
        rec["w_baseline"]*logit(baseline_test)+
        rec["w_banglabert"]*logit(bang_test)+
        rec["w_xlmr"]*logit(xlmr_test)
    )

threshold=float(np.median(best["cv"]["thresholds"]))
pred=(best_test>=threshold).astype(np.int8)

submission=champion.copy()
before=submission[TARGET].to_numpy(int)
changed=int((before!=pred).sum())

if passes:
    submission[TARGET]=pred
else:
    # No replacement: preserve V14 exactly.
    changed=0

submission_path=RUN_DIR/"submission_v17_best_toxic.csv"
submission.to_csv(submission_path,index=False)

assert submission["ID"].tolist()==champion["ID"].tolist()
assert submission[ALL_LABELS].isin([0,1]).all().all()

report={
    "version":"v17_is_toxic_oof_ensemble",
    "champion":"V14",
    "champion_kaggle_score":0.586,
    "baseline_crossfit_macro_f1":baseline_cv["score"],
    "selected":rec,
    "passes_gate":passes,
    "final_threshold":threshold,
    "test_rows_changed":changed,
    "test_toxic_positives_before":int(before.sum()),
    "test_toxic_positives_after":int(pred.sum()) if passes else int(before.sum()),
    "warning":"OOF gains are development evidence, not a guaranteed Kaggle improvement."
}
(RUN_DIR/"v17_report.json").write_text(json.dumps(report,indent=2))

print("\nRECOMMENDATION")
if passes:
    print("Candidate PASSED the conservative gate.")
    print("Send v17_results.zip to ChatGPT before Kaggle submission.")
    print("Changed test rows:",changed)
else:
    print("No candidate passed the gate. Keep V14 = 0.586.")


In [ ]:

# 12. Package results
zip_path=Path("/content/v17_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)
from google.colab import files
files.download(str(zip_path))
